#Propuesta 1. Features agregadas por producto

In [ ]:
product_features = df.groupby("name").agg(
    total_reviews=("reviews.rating", "count"),
    avg_rating=("reviews.rating", "mean"),
    min_rating=("reviews.rating", "min"),
    max_rating=("reviews.rating", "max"),
    negative_reviews=("is_negative_review", "sum"),
    neutral_reviews=("is_neutral_review", "sum"),
    positive_reviews=("is_positive_review", "sum"),
    not_recommended_count=("not_recommended", "sum"),
    avg_helpful_votes=("reviews.numHelpful", "mean"),
    total_helpful_votes=("reviews.numHelpful", "sum"),
    avg_review_length_words=("review_length_words", "mean"),
    first_review_date=("reviews.date", "min"),
    last_review_date=("reviews.date", "max")
).reset_index()

# =========================
# Añadir Brand y Primary Categories a product_features
# =========================
# Estas columnas son atributos a nivel de producto.
product_brand_category = df[['name', 'brand', 'primaryCategories']].drop_duplicates(subset=['name'])

# Fusionar con product_features
product_features = product_features.merge(
    product_brand_category,
    on='name',
    how='left'
)

# Ratios por producto
product_features["negative_review_rate"] = (
    product_features["negative_reviews"] / product_features["total_reviews"]
)

product_features["positive_review_rate"] = (
    product_features["positive_reviews"] / product_features["total_reviews"]
)

product_features["not_recommended_rate"] = (
    product_features["not_recommended_count"] / product_features["total_reviews"]
)

# Antigüedad en días
product_features["review_period_days"] = (
    product_features["last_review_date"] - product_features["first_review_date"]
).dt.days

# Frecuencia de reseñas
product_features["review_frequency"] = (
    product_features["total_reviews"] /
    product_features["review_period_days"].replace(0, np.nan)
)

print("FEATURES POR PRODUCTO")
print("Shape:", product_features.shape)
display(product_features.head())

FEATURES POR PRODUCTO
Shape: (65, 21)


,name,total_reviews,avg_rating,min_rating,max_rating,negative_reviews,neutral_reviews,positive_reviews,not_recommended_count,avg_helpful_votes,...,avg_review_length_words,first_review_date,last_review_date,brand,primaryCategories,negative_review_rate,positive_review_rate,not_recommended_rate,review_period_days,review_frequency
0,"All-New Fire 7 Tablet with Alexa, 7"" Display, ...",82,4.585366,1,5,2,2,78,3,0.573171,...,25.853659,2017-06-22 00:00:00+00:00,2018-04-26 00:00:00+00:00,Amazon,Electronics,0.024390,0.951220,0.036585,308.0,0.266234
1,"All-New Fire HD 8 Kids Edition Tablet, 8 HD Di...",233,4.630901,1,5,9,10,214,12,0.454936,...,41.000000,2017-06-15 00:00:00+00:00,2018-09-10 00:32:46+00:00,Amazon,Electronics,0.038627,0.918455,0.051502,452.0,0.515487
2,"All-New Fire HD 8 Kids Edition Tablet, 8 HD Di...",293,4.641638,1,5,6,11,276,9,0.313993,...,34.044369,2017-06-21 00:00:00+00:00,2018-05-25 00:00:00+00:00,Amazon,Electronics,0.020478,0.941980,0.030717,338.0,0.866864
3,"All-New Fire HD 8 Tablet with Alexa, 8 HD Disp...",883,4.578709,1,5,15,45,823,34,0.285391,...,31.682899,2017-06-20 00:00:00+00:00,2018-05-26 00:00:00+00:00,Amazon,Electronics,0.016988,0.932050,0.038505,340.0,2.597059
4,"All-New Fire HD 8 Tablet with Alexa, 8 HD Disp...",160,4.600000,1,5,5,2,153,9,0.731250,...,35.906250,2017-06-07 00:00:00+00:00,2018-05-22 00:00:00+00:00,Amazon,Electronics,0.031250,0.956250,0.056250,349.0,0.458453


##Propuesta 2: Features temporales producto-mes

In [ ]:
monthly_product = df.groupby(["name", "year_month"]).agg(
    monthly_reviews=("reviews.rating", "count"),
    monthly_avg_rating=("reviews.rating", "mean"),
    monthly_negative_reviews=("is_negative_review", "sum"),
    monthly_positive_reviews=("is_positive_review", "sum"),
    monthly_not_recommended=("not_recommended", "sum"),
    monthly_avg_helpful=("reviews.numHelpful", "mean"),
    monthly_avg_length_words=("review_length_words", "mean")
).reset_index()

# =========================
# 3. Añadir Brand y Primary Categories
# =========================
# Estas variables son a nivel de producto y se pueden añadir directamente.
monthly_product = monthly_product.merge(
    product_features[['name', 'brand', 'primaryCategories']],
    on='name',
    how='left'
)

# Ratios mensuales
monthly_product["monthly_negative_rate"] = (
    monthly_product["monthly_negative_reviews"] / monthly_product["monthly_reviews"]
)

monthly_product["monthly_positive_rate"] = (
    monthly_product["monthly_positive_reviews"] / monthly_product["monthly_reviews"]
)

monthly_product["monthly_not_recommended_rate"] = (
    monthly_product["monthly_not_recommended"] / monthly_product["monthly_reviews"]
)

# Orden temporal
monthly_product["year_month_date"] = pd.to_datetime(monthly_product["year_month"])
monthly_product = monthly_product.sort_values(["name", "year_month_date"])

# Lags
monthly_product["prev_month_avg_rating"] = (
    monthly_product.groupby("name")["monthly_avg_rating"].shift(1)
)

monthly_product["prev_month_negative_rate"] = (
    monthly_product.groupby("name")["monthly_negative_rate"].shift(1)
)

# Diferencias
monthly_product["rating_change"] = (
    monthly_product["monthly_avg_rating"] - monthly_product["prev_month_avg_rating"]
)

monthly_product["negative_rate_change"] = (
    monthly_product["monthly_negative_rate"] - monthly_product["prev_month_negative_rate"]
)

# Rolling windows de 3 meses
monthly_product["rolling_3m_avg_rating"] = (
    monthly_product.groupby("name")["monthly_avg_rating"]
    .transform(lambda x: x.rolling(window=3, min_periods=1).mean())
)

monthly_product["rolling_3m_negative_rate"] = (
    monthly_product.groupby("name")["monthly_negative_rate"]
    .transform(lambda x: x.rolling(window=3, min_periods=1).mean())
)

print("FEATURES PRODUCTO-MES")
print("Shape:", monthly_product.shape)
display(monthly_product.head(10))

FEATURES PRODUCTO-MES
Shape: (561, 21)


,name,year_month,monthly_reviews,monthly_avg_rating,monthly_negative_reviews,monthly_positive_reviews,monthly_not_recommended,monthly_avg_helpful,monthly_avg_length_words,brand,...,monthly_negative_rate,monthly_positive_rate,monthly_not_recommended_rate,year_month_date,prev_month_avg_rating,prev_month_negative_rate,rating_change,negative_rate_change,rolling_3m_avg_rating,rolling_3m_negative_rate
0,"All-New Fire 7 Tablet with Alexa, 7"" Display, ...",2017-06,1,5.000000,0,1,0,44.000000,102.000000,Amazon,...,0.00000,1.000000,0.000000,2017-06-01,NaN,NaN,NaN,NaN,5.000000,0.00000
1,"All-New Fire 7 Tablet with Alexa, 7"" Display, ...",2017-07,5,4.600000,0,5,0,0.200000,45.400000,Amazon,...,0.00000,1.000000,0.000000,2017-07-01,5.000000,0.00000,-0.400000,0.00000,4.800000,0.00000
2,"All-New Fire 7 Tablet with Alexa, 7"" Display, ...",2017-08,9,4.222222,0,8,1,0.222222,28.555556,Amazon,...,0.00000,0.888889,0.111111,2017-08-01,4.600000,0.00000,-0.377778,0.00000,4.607407,0.00000
3,"All-New Fire 7 Tablet with Alexa, 7"" Display, ...",2017-09,3,4.666667,0,3,0,0.000000,21.666667,Amazon,...,0.00000,1.000000,0.000000,2017-09-01,4.222222,0.00000,0.444444,0.00000,4.496296,0.00000
4,"All-New Fire 7 Tablet with Alexa, 7"" Display, ...",2017-10,1,5.000000,0,1,0,0.000000,25.000000,Amazon,...,0.00000,1.000000,0.000000,2017-10-01,4.666667,0.00000,0.333333,0.00000,4.629630,0.00000
5,"All-New Fire 7 Tablet with Alexa, 7"" Display, ...",2017-12,41,4.634146,2,39,2,0.000000,22.097561,Amazon,...,0.04878,0.951220,0.048780,2017-12-01,5.000000,0.00000,-0.365854,0.04878,4.766938,0.01626
6,"All-New Fire 7 Tablet with Alexa, 7"" Display, ...",2018-01,11,4.545455,0,11,0,0.000000,22.636364,Amazon,...,0.00000,1.000000,0.000000,2018-01-01,4.634146,0.04878,-0.088692,-0.04878,4.726534,0.01626
7,"All-New Fire 7 Tablet with Alexa, 7"" Display, ...",2018-02,5,4.800000,0,5,0,0.000000,26.600000,Amazon,...,0.00000,1.000000,0.000000,2018-02-01,4.545455,0.00000,0.254545,0.00000,4.659867,0.01626
8,"All-New Fire 7 Tablet with Alexa, 7"" Display, ...",2018-03,4,4.250000,0,3,0,0.000000,31.500000,Amazon,...,0.00000,0.750000,0.000000,2018-03-01,4.800000,0.00000,-0.550000,0.00000,4.531818,0.00000
9,"All-New Fire 7 Tablet with Alexa, 7"" Display, ...",2018-04,2,5.000000,0,2,0,0.000000,15.000000,Amazon,...,0.00000,1.000000,0.000000,2018-04-01,4.250000,0.00000,0.750000,0.00000,4.683333,0.00000


##Propuesta 3: Combinar features de producto con producto-mes

In [ ]:
final_dataset = monthly_product.merge(
    product_features,
    on="name",
    how="left"
)

print("DATASET FINAL COMBINADO")
print("Shape:", final_dataset.shape)
print("Columnas:", final_dataset.columns.tolist())
display(final_dataset.head())

DATASET FINAL COMBINADO
Shape: (561, 41)
Columnas: ['name', 'year_month', 'monthly_reviews', 'monthly_avg_rating', 'monthly_negative_reviews', 'monthly_positive_reviews', 'monthly_not_recommended', 'monthly_avg_helpful', 'monthly_avg_length_words', 'brand_x', 'primaryCategories_x', 'monthly_negative_rate', 'monthly_positive_rate', 'monthly_not_recommended_rate', 'year_month_date', 'prev_month_avg_rating', 'prev_month_negative_rate', 'rating_change', 'negative_rate_change', 'rolling_3m_avg_rating', 'rolling_3m_negative_rate', 'total_reviews', 'avg_rating', 'min_rating', 'max_rating', 'negative_reviews', 'neutral_reviews', 'positive_reviews', 'not_recommended_count', 'avg_helpful_votes', 'total_helpful_votes', 'avg_review_length_words', 'first_review_date', 'last_review_date', 'brand_y', 'primaryCategories_y', 'negative_review_rate', 'positive_review_rate', 'not_recommended_rate', 'review_period_days', 'review_frequency']


,name,year_month,monthly_reviews,monthly_avg_rating,monthly_negative_reviews,monthly_positive_reviews,monthly_not_recommended,monthly_avg_helpful,monthly_avg_length_words,brand_x,...,avg_review_length_words,first_review_date,last_review_date,brand_y,primaryCategories_y,negative_review_rate,positive_review_rate,not_recommended_rate,review_period_days,review_frequency
0,"All-New Fire 7 Tablet with Alexa, 7"" Display, ...",2017-06,1,5.000000,0,1,0,44.000000,102.000000,Amazon,...,25.853659,2017-06-22 00:00:00+00:00,2018-04-26 00:00:00+00:00,Amazon,Electronics,0.02439,0.95122,0.036585,308.0,0.266234
1,"All-New Fire 7 Tablet with Alexa, 7"" Display, ...",2017-07,5,4.600000,0,5,0,0.200000,45.400000,Amazon,...,25.853659,2017-06-22 00:00:00+00:00,2018-04-26 00:00:00+00:00,Amazon,Electronics,0.02439,0.95122,0.036585,308.0,0.266234
2,"All-New Fire 7 Tablet with Alexa, 7"" Display, ...",2017-08,9,4.222222,0,8,1,0.222222,28.555556,Amazon,...,25.853659,2017-06-22 00:00:00+00:00,2018-04-26 00:00:00+00:00,Amazon,Electronics,0.02439,0.95122,0.036585,308.0,0.266234
3,"All-New Fire 7 Tablet with Alexa, 7"" Display, ...",2017-09,3,4.666667,0,3,0,0.000000,21.666667,Amazon,...,25.853659,2017-06-22 00:00:00+00:00,2018-04-26 00:00:00+00:00,Amazon,Electronics,0.02439,0.95122,0.036585,308.0,0.266234
4,"All-New Fire 7 Tablet with Alexa, 7"" Display, ...",2017-10,1,5.000000,0,1,0,0.000000,25.000000,Amazon,...,25.853659,2017-06-22 00:00:00+00:00,2018-04-26 00:00:00+00:00,Amazon,Electronics,0.02439,0.95122,0.036585,308.0,0.266234
